In [26]:
## import statements here
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
## function for one-hot encoding
def one_hot(col_list, df):

    encoder = OneHotEncoder(sparse_output=False)

    one_hot_encoded = encoder.fit_transform(df[col_list])

    one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(col_list))

    df_encoded = pd.concat([df, one_hot_df], axis=1)

    df_encoded = df_encoded.drop(col_list, axis=1)

    return df_encoded

In [3]:
## function for replacing filled values with 1 and NaN with 0
def fill_binary(col_list, df):
    
    for cols in col_list:
        col_name = "bin_" + cols

        df[col_name] = np.where(df[cols].notna(), 1, 0)

    df = df.drop(columns=col_list)

    return df

In [4]:
## lists to drop

## columns to drop because they are covered by the lat/long information
geo_validated = ['incident_zip', 'incident_address', "street_name", "cross_street_1", "cross_street_2", "intersection_street_1", 
                 "intersection_street_2", "location", "community_board", "bbl", "x_coordinate_state_plane", "y_coordinate_state_plane"]

## columns to drop because they aren't available when the complaint is submitted
post_complaint = ["due_date", "resolution_action_updated_date", "resolution_description"]

## columns we are dropping for other reasons
other_drop = ["agency_name", "descriptor_2", "bridge_highway_direction", "road_ramp", "bridge_highway_segment", "source_parquet", 
              "taxi_pick_up_location", "park_borough", "vehicle_type", "taxi_company_borough" ]

In [16]:
## bring in the data
file_loc = "C:/Users/esoph/OneDrive/Documents/MSDS_498/Data/nyc_311_all_appended.parquet"

df = pd.read_parquet(file_loc, engine="fastparquet")

##df.head()

In [17]:
## fixing the date errors
## adding date information 

df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')
df['year'] = df['created_date'].dt.year
df['resolution_time_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600


In [18]:
## dropping the unneeded columns
df_dropped_1 = df.drop(columns=geo_validated)

df_dropped_2 = df_dropped_1.drop(columns=post_complaint)

df_dropped_3 = df_dropped_2.drop(columns=other_drop)

In [19]:
# filter to just the closed incidents
df_closed = df_dropped_3[df_dropped_3['status'] == "Closed"]


In [20]:
# drop the computation columns
compute_cols = ["status", "closed_date"]

df_compute = df_closed.drop(columns=compute_cols)

# change "Unspecified" in the park_facility_name column to None
df_compute = df_compute.replace('Unspecified', None)

In [21]:
## change the lat/long and resolution_time_hours into float64
cols = ['latitude', 'longitude', "resolution_time_hours"]
df_compute[cols] = df_compute[cols].fillna(0).astype('float64')

In [22]:
## make the binary columns into 1s and 0s
bin_cols = ["bridge_highway_name", "park_facility_name", "landmark"]

binary_df = fill_binary(bin_cols, df_compute)

In [23]:
## make sure all of the dataframe rows have "None" instead of None (obejct type)
df_none = binary_df.fillna('none')


In [31]:
## one-hot encoding
cat_cols = df_none.select_dtypes(include=['object']).columns.tolist()

df_encoded = pd.get_dummies(df_none, columns=cat_cols)

In [32]:
df_encoded.head(3).T

,1,4,27
unique_key,68557159,68551885,68548189
created_date,2026-04-04 02:54:42,2026-04-04 02:01:53,2026-04-04 01:44:00
latitude,40.741714,40.699938,40.821912
longitude,-73.906332,-73.911808,-73.956542
year,2026,2026,2026
...,...,...,...
open_data_channel_type_MOBILE,False,False,False
open_data_channel_type_ONLINE,False,False,False
open_data_channel_type_OTHER,False,False,False
open_data_channel_type_PHONE,False,False,True
